Human in the loop

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool
from langchain_groq import ChatGroq

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"


llm = ChatGroq(
    model="openai/gpt-oss-120b",
)

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [2]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='5ddbd504-f814-4fdb-9223-07e2e418bc7b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use function send_email. Provide body, subject, to. The user didn\'t specify subject or body details. We can ask for clarification? Probably we should ask for more details. But we could assume a generic email: Subject: Q4 Results, Body: Provide summary. But better to ask for specifics. The instruction: "Send an email to team@company.com about the Q4 results". We can assume they want an email. Provide a generic email. Let\'s craft: Subject: Q4 Results Overview, Body: Dear Team, ... Provide brief. Use function.', 'tool_calls': [{'id': 'fc_2260e843-95ad-4e6d-abf5-d6e6ed16d77a', 'function': {'arguments': '{"body":"Dear Team,\\n\\nPlease find attached the summary of our Q4 results. 

In [3]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email has been sent to **team@company.com** with the subject **“Q4 Results Overview.”** Let me know if you’d like any changes, additional details, or an attachment included.


In [4]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I’m ready to help with that, but deleting records is an irreversible operation. Could you please confirm that you’d like to permanently remove **all** rows from the `users` table where `active = false`? Once you confirm, I’ll go ahead and execute the deletion.
